# Experiment 33 — Amazon parallel scaling + corrected quality rerun

Companion to the ML-1M streaming-speed work. This notebook covers **Beauty, Video Games, Sports, and Toys**.

Part 1 measures matched A100/BF16 training throughput for **StreamingSparseWalker+2T vs SASRec** at batch 128/256/512/1024. Histories are capped to 50 prediction positions and both models use FullCE.

Part 2 is optional and reruns the corrected **SparseWalker v1.1 vs SASRec** full-catalog quality comparison under one common Amazon protocol, to verify the older 'Walker wins on Amazon' result after the recurrence fixes.


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import os, sys, shutil, subprocess, json, runpy, torch
from pathlib import Path

REPO='/content/Sparsewalker'
BRANCH='agent/streaming-training-speed-v3'
if os.path.exists(REPO): shutil.rmtree(REPO)
subprocess.run(['git','clone','-q','-b',BRANCH,'https://github.com/hanialshater/Sparsewalker-.git',REPO], check=True)
for p in [f'{REPO}/src', f'{REPO}/experiments', f'{REPO}/benchmarks']:
    if p not in sys.path: sys.path.insert(0,p)
import sparsewalker
assert torch.cuda.is_available(), 'GPU runtime required'
print('GPU',torch.cuda.get_device_name(0),'torch',torch.__version__,'bf16',torch.cuda.is_bf16_supported())
print('BRANCH',BRANCH,'PACKAGE',sparsewalker.__file__)


## 1. Training parallelism sweep

Runs 24 length-bucketed batches per cell. The main outputs are valid `positions_per_s`, peak allocated GPU memory, and Walker/SASRec throughput ratio. OOM cells are recorded rather than aborting the sweep.


In [ ]:
SCRIPT=f'{REPO}/experiments/run_amazon_parallel_scaling.py'
sys.argv=[SCRIPT,
    '--datasets','beauty,video_games,sports,toys',
    '--batch-sizes','128,256,512,1024',
    '--benchmark-batches','24',
]
runpy.run_path(SCRIPT, run_name='__main__')


In [ ]:
import pandas as pd
p=Path('/content/drive/MyDrive/sparsewalker_amazon_parallel_scaling/result.json')
rows=json.loads(p.read_text())
df=pd.DataFrame(rows)
cols=['dataset','model','batch_size','positions_per_s','seconds','peak_allocated_GB','status']
display(df[[c for c in cols if c in df.columns]].sort_values(['dataset','batch_size','model']))

ok=df[df.status.eq('OK')].copy()
if len(ok):
    pivot=ok.pivot_table(index=['dataset','batch_size'],columns='model',values='positions_per_s',aggfunc='first').reset_index()
    if 'SASRec' in pivot.columns and 'StreamingSparseWalker+2T' in pivot.columns:
        pivot['walker_vs_sasrec_ratio']=pivot['StreamingSparseWalker+2T']/pivot['SASRec']
    display(pivot)


## 2. Optional corrected Amazon quality rerun

This is slower. It trains **SASRec and corrected SparseWalker v1.1** with the same FullCE objective, max_len=50, leave-two-out split, full-catalog evaluation, and seen-item masking. Default is 30 epochs with validation every 2 epochs and patience 10.

Set `RUN_QUALITY=True` when you want to re-establish the Amazon quality table.


In [ ]:
RUN_QUALITY=False
if RUN_QUALITY:
    QSCRIPT=f'{REPO}/experiments/run_amazon_quality_pair.py'
    sys.argv=[QSCRIPT,
        '--datasets','beauty,video_games,sports,toys',
        '--models','SASRec,SparseWalker',
        '--epochs','30',
        '--eval-every','2',
        '--patience','10',
        '--batch-size','512',
    ]
    runpy.run_path(QSCRIPT, run_name='__main__')


In [ ]:
q=Path('/content/drive/MyDrive/sparsewalker_amazon_quality_v11/summary_seed42.json')
if q.exists():
    qr=pd.DataFrame(json.loads(q.read_text()))
    display(qr[['dataset','model','best_epoch','best_val_NDCG@10','NDCG@10','HR@10','MRR@10','params']].sort_values(['dataset','model']))
    wide=qr.pivot(index='dataset',columns='model',values='NDCG@10')
    if 'SASRec' in wide.columns and 'SparseWalker' in wide.columns:
        wide['Walker_minus_SASRec']=wide['SparseWalker']-wide['SASRec']
        wide['Walker_lift_pct']=100*(wide['SparseWalker']/wide['SASRec']-1)
    display(wide)
else:
    print('No quality summary yet; set RUN_QUALITY=True above when ready.')


## Interpretation

- If Walker throughput keeps scaling through 512/1024 on Amazon, ordinary user-batch parallelism is enough for these short-history datasets.
- The SASRec rows give the matched training-speed gap we were missing.
- The optional v1.1 quality table tells us whether the historical Amazon advantage survives the recurrence fixes.
- Keep this quality result separate from paper-protocol reproductions; this notebook is an internal common-protocol comparison.
